Paragraph 1 — pick your files and write down which ones.
Go to the COW website. Download the exact files you're going to use. Write down the version number and note that the incident files only cover 1993–2014. Don't lose track of which version you used.
Then a warning: this data is old. Whatever you find is about 1993–2014. Don't claim it says anything about how conflicts work today.
Paragraph 2 — build the list of cases and show it works.
Make a file of every country pair that qualifies, and count them: how many cases, how many turned violent, and how many fall in each chunk when you split the data up.
Prove four things can actually be pulled out of the raw data: who was fighting whom, whether it started with a threat rather than force, when things happened, and whether force came later.
And check the disputes with lots of countries in them. If sixteen countries act against one, don't just assume all sixteen were really in a fight with it. Look at them.
Paragraph 3 — pick a paper and copy it before changing anything.
Find a published paper that used data you can also get. Redo their analysis exactly as they did it, and see if you get their numbers.
Only after that, redo it your way with the seven-day and ninety-day rules. Report those as two separate results. Don't blend them together.


# Team 2 — Cohort Builder
MID 5.01 incident data. Unit = country pair inside a dispute.

**A pair qualifies if all four are true:**
1. They interacted (one acted, other was target, same incident)
2. Their first interaction has a real date
3. That first interaction was `hostlev` 2 or 3 (threat / show of force), not 4+
4. No force between them in the next 7 days

**Outcome:** force (`hostlev` >= 4) between them in days 8–97.

In [3]:
!pwd
!ls

/Users/jerryluo/Desktop/rad_intel_2_test_1
MID 5.0 Codebook.pdf MIDIP_5.01.dta       MIDI_5.01.dta
MIDIP_5.01.csv       MIDI_5.01.csv        cohort_builder.ipynb


In [5]:
import pandas as pd, numpy as np, hashlib
pd.set_option('display.width', 120)

UP = ''
midi  = pd.read_csv(UP + 'MIDI_5.01.csv')
midip = pd.read_csv(UP + 'MIDIP_5.01.csv')

for f in ['MIDI_5.01.csv', 'MIDIP_5.01.csv']:
    print(f, hashlib.md5(open(UP+f,'rb').read()).hexdigest())

MIDI_5.01.csv aa05467ecec23fa7c057b10e2a3b50ea
MIDIP_5.01.csv 92020d013943b7916134b244bdcf0430


## 1. Dates
Dates are split across three columns and missing values are `-9`.
Build one real date column; leave it as `NaT` when the day or month is missing.

In [6]:
def mkdate(df):
    ok = (df.stday > 0) & (df.stmon > 0)
    d = pd.to_datetime(dict(year=df.styear,
                            month=df.stmon.where(ok, 1),
                            day=df.stday.where(ok, 1)), errors='coerce')
    return d.where(ok)

midip['date'] = mkdate(midip)
print('incidents missing a usable date:', midip.date.isna().mean().round(3))

incidents missing a usable date: 0.044


## 2. Build country pairs
In each incident, `insidea == 1` are the countries that acted and `insidea == 0` are the targets.
Cross-join actors against targets to get the pairs.

**Caveat:** if 16 countries act against 1, this makes 16 pairs. Whether all 16 really
interacted is the multiparty question — flagged below, not solved here.

In [7]:
a = midip[midip.insidea == 1][['dispnum','ord_incidnum','date','ccode','hostlev']]
t = midip[midip.insidea == 0][['dispnum','ord_incidnum','ccode']]

pairs = a.merge(t, on=['dispnum','ord_incidnum'], suffixes=('_actor','_target'))
pairs = pairs[pairs.ccode_actor != pairs.ccode_target]

pairs['c_lo'] = pairs[['ccode_actor','ccode_target']].min(axis=1)
pairs['c_hi'] = pairs[['ccode_actor','ccode_target']].max(axis=1)
pairs['dyad'] = (pairs.dispnum.astype(str) + '_' +
                 pairs.c_lo.astype(str) + '_' + pairs.c_hi.astype(str))

# how many countries acted in each incident — the multiparty flag
pairs['n_actors'] = pairs.groupby('ord_incidnum').ccode_actor.transform('nunique')

print(len(pairs), 'pair-incident rows |', pairs.dyad.nunique(), 'distinct dispute-pairs')
pairs.head()

5137 pair-incident rows | 941 distinct dispute-pairs


,dispnum,ord_incidnum,date,ccode_actor,hostlev,ccode_target,c_lo,c_hi,dyad,n_actors
0,3551,3551001,1992-07-16,211,3,345,211,345,3551_211_345,13
1,3551,3551001,1992-07-16,20,3,345,20,345,3551_20_345,13
2,3551,3551001,1992-07-16,390,3,345,345,390,3551_345_390,13
3,3551,3551001,1992-07-16,220,3,345,220,345,3551_220_345,13
4,3551,3551001,1992-07-16,255,3,345,255,345,3551_255_345,13


## 3. Apply the four rules
One row per pair. `status` records why anything was dropped — never drop silently.

In [8]:
LANDMARK, HORIZON = 7, 90
rows = []

for dyad, g in pairs.groupby('dyad'):
    g = g.sort_values('ord_incidnum')
    dated = g[g.date.notna()]

    if len(dated) == 0:
        rows.append({'dyad': dyad, 'status': 'no_dated_incident'});      continue
    if pd.isna(g.iloc[0].date):
        rows.append({'dyad': dyad, 'status': 'first_incident_undated'}); continue

    first, t0 = dated.iloc[0], dated.iloc[0].date
    if first.hostlev >= 4:
        rows.append({'dyad': dyad, 'status': 'starts_at_force'});        continue

    wk = dated[(dated.date > t0) & (dated.date <= t0 + pd.Timedelta(days=LANDMARK))]
    if (wk.hostlev >= 4).any():
        rows.append({'dyad': dyad, 'status': 'force_within_landmark'});  continue

    fut = dated[(dated.date >  t0 + pd.Timedelta(days=LANDMARK)) &
                (dated.date <= t0 + pd.Timedelta(days=LANDMARK + HORIZON))]
    win = dated[dated.date <= t0 + pd.Timedelta(days=LANDMARK)]
    sides = set(zip(win.ccode_actor, win.ccode_target))

    rows.append({
        'dyad'           : dyad,
        'status'         : 'eligible',
        't0'             : t0,
        'year'           : t0.year,
        'escalated'      : int((fut.hostlev >= 4).any()),
        'init_sev'       : int(first.hostlev),                          # baseline
        'n_incidents_wk' : len(win),                                    # baseline
        'days_to_last_wk': int((win.date.max() - t0).days),             # recency
        'reciprocated_wk': int(any((b, a) in sides for a, b in sides)), # THE EXTENSION
        'multiparty'     : int(g.n_actors.max() > 1),
    })

cohort = pd.DataFrame(rows)
cohort.status.value_counts()

status
eligible                  520
starts_at_force           333
no_dated_incident          39
first_incident_undated     31
force_within_landmark      18
Name: count, dtype: int64

## 4. Counts — this is the feasibility deliverable

In [9]:
e = cohort[cohort.status == 'eligible'].copy()
print(f'eligible: {len(e)}   escalations: {int(e.escalated.sum())}   rate: {e.escalated.mean():.1%}')

print('\nby multiparty:')
print(e.groupby('multiparty').escalated.agg(['size','sum','mean']))

def bucket(y): return 'train' if y < 2004 else ('dev' if y < 2010 else 'final')
e['split'] = e.year.map(bucket)
print('\nchronological split:')
print(e.groupby('split').escalated.agg(['size','sum','mean']))

eligible: 520   escalations: 38   rate: 7.3%

by multiparty:
            size   sum      mean
multiparty                      
0.0          372  33.0  0.088710
1.0          148   5.0  0.033784

chronological split:
       size   sum      mean
split                      
dev      87   9.0  0.103448
final   121   7.0  0.057851
train   312  22.0  0.070513


## 5. Drop multiparty (Braithwaite & Lemke's approach)
They kept only disputes that were bilateral on day one. Run this to see the cost.

In [ ]:
bi = e[e.multiparty == 0]
print(f'bilateral only: {len(bi)}   escalations: {int(bi.escalated.sum())}   rate: {bi.escalated.mean():.1%}')
print(bi.groupby('split').escalated.agg(['size','sum','mean']))

## TODO for whoever owns data
- Same-day ordering: incidents on the same date have no tiebreak
- Repeat disputes between the same pair are separate rows — check for dependence
- Disputes still running at end of 2014 need censoring
- Decide the multiparty rule and write it down before looking at results

In [ ]:
cohort.to_csv('cohort.csv', index=False)
pairs.to_csv('pair_incidents.csv', index=False)
print('saved')